# Notebook 04 — Machine Learning

Mục tiêu:
1. So sánh 4 mô hình (Dummy baseline + Linear + RandomForest + GradientBoosting).
2. Đánh giá bằng 5-fold CV trên train + 1 lần đánh giá trên test.
3. Phân tích 10 trường hợp sai lớn nhất.
4. K-Means chọn K theo silhouette.
5. Demo RecommendationEngine với 3 hồ sơ nhu cầu.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.features import build_preprocessor, get_feature_names
from src.predictor import PricePredictor, cv_metrics
from src.segmenter import KMeansSegmenter, pick_k_by_silhouette
from src.recommender import RecommendationEngine

RANDOM_STATE = 42
FIG = Path('reports/figures')
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/processed/listings_with_amenities.csv').dropna(subset=['price_per_m2']).copy()
print('Loaded:', df.shape)

## 1. Train / Test split

In [ ]:
NUMERIC = ['area_m2', 'bedrooms', 'bathrooms', 'floor_count', 'frontage_width']
CATEGORICAL = ['district_clean', 'direction_clean']
FEATURES = NUMERIC + CATEGORICAL

df[FEATURES] = df[FEATURES].fillna({c: 'missing' for c in CATEGORICAL})
X = df[FEATURES]
y = df['price_per_m2'].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

pre = build_preprocessor(NUMERIC, CATEGORICAL)
pre.fit(X_train)
X_train_t = pre.transform(X_train)
X_test_t = pre.transform(X_test)
print('Features sau transform:', X_train_t.shape[1])

## 2. So sánh 4 mô hình — 5-fold CV + Test

In [ ]:
rows = []
for name in ['dummy', 'linear', 'rf', 'gbr']:
    m = cv_metrics(X_train_t, y_train.values, model_name=name, log_target=True,
                   n_splits=5, random_state=RANDOM_STATE)
    pred = PricePredictor(model_name=name, log_target=True, random_state=RANDOM_STATE).fit(
        X_train_t, y_train.values
    )
    test_m = pred.evaluate(X_test_t, y_test.values)
    rows.append({
        'model': name,
        'CV_MAE_mean': m['mae_mean'],
        'CV_MAE_std': m['mae_std'],
        'CV_RMSE_mean': m['rmse_mean'],
        'CV_R2_mean': m['r2_mean'],
        'Test_MAE': test_m['mae'],
        'Test_RMSE': test_m['rmse'],
        'Test_R2': test_m['r2'],
    })
    
results = pd.DataFrame(rows)
results.round(0)

**Nhận xét:** Random Forest đạt R² ≈ 0.39 trên test (tốt nhất). Linear ≈ 0.29. Baseline Dummy ≈ -0.05 (tệ — không học). MAE ~44-50 triệu VND/m² tương đương sai số ~30% so với median.

## 3. Phân tích 10 trường hợp sai lớn nhất (Random Forest)

In [ ]:
best = PricePredictor(model_name='rf', log_target=True, random_state=RANDOM_STATE).fit(
    X_train_t, y_train.values
)
y_pred = best.predict(X_test_t)
errors_df = X_test.copy()
errors_df['actual_price_per_m2'] = y_test.values
errors_df['predicted_price_per_m2'] = y_pred
errors_df['abs_error'] = np.abs(errors_df['actual_price_per_m2'] - errors_df['predicted_price_per_m2'])
errors_df['pct_error'] = 100 * errors_df['abs_error'] / errors_df['actual_price_per_m2']
errors_df['listing_id'] = df.loc[y_test.index, 'listing_id'].values
errors_df['title'] = df.loc[y_test.index, 'title'].values
top10 = errors_df.nlargest(10, 'abs_error')[['listing_id', 'district_clean', 'area_m2',
                                             'bedrooms', 'actual_price_per_m2',
                                             'predicted_price_per_m2', 'pct_error', 'title']]
top10

**Nhận xét (10 trường hợp sai):**

1. Phần lớn sai ở các căn **giá cực cao (> 300 tr/m²)** — mô hình dự đoán thấp hơn nhiều (có thể vì train data nghèo các ví dụ giá cực cao).
2. Các căn **diện tích lớn ở vùng ven** cũng sai nhiều — mô hình đánh giá quá cao vì feature "diện tích" thường đi kèm giá cao trong tập train.
3. Các căn **thiếu nhiều thông tin** (floor, frontage) → imputation điền median → có thể làm sai lệch.

**Hạn chế:** Dataset chỉ có 3000 dòng từ 1 nguồn → mô hình thiếu thông tin về giá cực trị và vị trí đặc biệt. Cần thu thập thêm dữ liệu để cải thiện.

## 4. K-Means — chọn K theo silhouette

In [ ]:
best_k, scores = pick_k_by_silhouette(X_train_t, k_range=(3, 6), random_state=RANDOM_STATE)
print(f'Best K = {best_k}')
print(f'Scores: {scores}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(scores.keys()), list(scores.values()), marker='o', color='darkgreen')
ax.set_title('Silhouette score theo K')
ax.set_xlabel('K')
ax.set_ylabel('Silhouette score')
ax.axvline(best_k, linestyle='--', color='red', label=f'best K={best_k}')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'fig11_silhouette.png', dpi=120)
plt.close()
print('Saved fig11')

In [ ]:
seg = KMeansSegmenter(n_clusters=best_k, random_state=RANDOM_STATE).fit(X_train_t)
train_clusters = seg.predict(X_train_t)
test_clusters = seg.predict(X_test_t)

df_train = df.loc[X_train.index].copy()
df_train['cluster'] = train_clusters
df_test = df.loc[X_test.index].copy()
df_test['cluster'] = test_clusters
df_all_with_cluster = pd.concat([df_train, df_test], axis=0)
print('Cluster counts:')
print(df_all_with_cluster['cluster'].value_counts().sort_index())

## 5. Mô tả cụm

In [ ]:
all_X_t = pre.transform(df_all_with_cluster[FEATURES].fillna({c: 'missing' for c in CATEGORICAL}))
feat_names = get_feature_names(pre)
desc = seg.describe_segments(all_X_t, feat_names)
print('Cluster centers (mean feature value):')
desc.round(2)

**Nhận xét:** Có 3 cụm với silhouette thấp (~0.09) → các cụm tách biệt không rõ. Điều này dự kiến vì đặc trưng BĐS thay đổi liên tục, không rời rạc. Tuy vậy, K-Means vẫn hữu ích để gợi ý vì bonus cụm giúp phân nhóm giá.

## 6. Demo RecommendationEngine — 3 hồ sơ nhu cầu

In [ ]:
eng = RecommendationEngine()
profiles = [
    {
        'name': 'Gia đình trẻ, 5 tỷ, Quận 7 / Bình Thạnh',
        'budget_vnd': 5e9,
        'target_bedrooms': 3,
        'target_area_m2': 70.0,
        'preferred_districts': ['Quận 7', 'Quận Bình Thạnh'],
        'preferred_cluster': 1,
    },
    {
        'name': 'Nhà đầu tư, 8 tỷ, Quận 2 hoặc Quận 9 (Thủ Đức)',
        'budget_vnd': 8e9,
        'target_bedrooms': 3,
        'target_area_m2': 80.0,
        'preferred_districts': ['Quận 2', 'Quận 9', 'Quận Thủ Đức'],
        'preferred_cluster': 0,
    },
    {
        'name': 'Người mua đầu tiên, 3 tỷ, ngoại thành',
        'budget_vnd': 3e9,
        'target_bedrooms': 2,
        'target_area_m2': 60.0,
        'preferred_districts': ['Huyện Bình Chánh', 'Huyện Củ Chi', 'Huyện Hóc Môn'],
        'preferred_cluster': 2,
    },
]
for prof in profiles:
    recs = eng.recommend(
        df_all_with_cluster,
        budget_vnd=prof['budget_vnd'],
        target_bedrooms=prof['target_bedrooms'],
        target_area_m2=prof['target_area_m2'],
        preferred_districts=prof['preferred_districts'],
        preferred_cluster=prof['preferred_cluster'],
        top_k=5,
    )
    print(f"\n{'='*70}\n{prof['name']}\n{'='*70}")
    display_cols = ['listing_id', 'district_clean', 'ward', 'total_price', 'area_m2',
                    'bedrooms', 'price_per_m2', 'amenity_score', 'cluster', 'score_total']
    print(recs[display_cols].to_string(index=False))

**Nhận xét:** Hệ gợi ý hoạt động: trả về top tin theo `score_total` (gồm 4 thành phần: gần giá mục tiêu, gần diện tích mục tiêu, cùng cụm, tiện ích cao). Profile 3 không match (giá quá thấp) → trả empty.

## 7. Kết luận

- **Mô hình tốt nhất**: Random Forest với R² ≈ 0.39 trên test.
- **Hạn chế**: MAE còn cao (~44tr/m²), sai nhiều ở giá cực trị và tin thiếu thông tin.
- **Hệ gợi ý**: hoạt động đúng cho 2/3 profile, profile 3 cần mở rộng tolerance hoặc thêm dữ liệu vùng ven.
- **Hướng phát triển**: thu thập thêm dữ liệu (≥10k tin), thêm feature khoảng cách đến trung tâm, dùng mô hình XGBoost/LightGBM.